In [ ]:
# DBN (Single Cell)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# ---------------- SETUP ----------------
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

batch_size = 128
pretrain_epochs = 5
finetune_epochs = 10
lr_rbm = 0.01
lr_ft = 1e-3

# ---------------- DATA ----------------
transform = transforms.ToTensor()

train_ds = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

# ---------------- RBM ----------------
class RBM(nn.Module):
    def __init__(self, v, h):
        super().__init__()
        self.W = nn.Parameter(torch.randn(v, h) * 0.01)
        self.vb = nn.Parameter(torch.zeros(v))
        self.hb = nn.Parameter(torch.zeros(h))

    def v_to_h(self, v):
        p = torch.sigmoid(v @ self.W + self.hb)
        return p, torch.bernoulli(p)

    def h_to_v(self, h):
        p = torch.sigmoid(h @ self.W.t() + self.vb)
        return p, torch.bernoulli(p)

# ---------------- PRETRAIN RBM1 ----------------
rbm1 = RBM(784, 256).to(device)

for epoch in range(pretrain_epochs):
    for x, _ in train_loader:
        v0 = x.view(-1,784).to(device)
        ph0, h0 = rbm1.v_to_h(v0)
        vk_prob, vk = rbm1.h_to_v(h0)
        phk, _ = rbm1.v_to_h(vk)

        with torch.no_grad():
            bs = v0.size(0)
            rbm1.W += lr_rbm * ((v0.t() @ ph0) - (vk.t() @ phk)) / bs
            rbm1.vb += lr_rbm * torch.mean(v0 - vk, dim=0)
            rbm1.hb += lr_rbm * torch.mean(ph0 - phk, dim=0)

    print("RBM1 epoch", epoch+1)

# ---------------- CREATE HIDDEN DATA ----------------
def transform_data(rbm, loader):
    feats, labels = [], []
    with torch.no_grad():
        for x,y in loader:
            v = x.view(-1,784).to(device)
            h,_ = rbm.v_to_h(v)
            feats.append(h.cpu())
            labels.append(y)
    return TensorDataset(torch.cat(feats), torch.cat(labels))

train_h1 = transform_data(rbm1, train_loader)
loader_h1 = DataLoader(train_h1, batch_size=batch_size, shuffle=True)

# ---------------- PRETRAIN RBM2 ----------------
rbm2 = RBM(256,128).to(device)

for epoch in range(pretrain_epochs):
    for x,_ in loader_h1:
        v0 = x.to(device)
        ph0,h0 = rbm2.v_to_h(v0)
        vk_prob,vk = rbm2.h_to_v(h0)
        phk,_ = rbm2.v_to_h(vk)

        with torch.no_grad():
            bs = v0.size(0)
            rbm2.W += lr_rbm*((v0.t()@ph0)-(vk.t()@phk))/bs
            rbm2.vb += lr_rbm*torch.mean(v0-vk,0)
            rbm2.hb += lr_rbm*torch.mean(ph0-phk,0)

    print("RBM2 epoch", epoch+1)

# ---------------- DBN ----------------
class DBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784,256)
        self.fc2 = nn.Linear(256,128)
        self.fc3 = nn.Linear(128,10)

        with torch.no_grad():
            self.fc1.weight.copy_(rbm1.W.t())
            self.fc1.bias.copy_(rbm1.hb)
            self.fc2.weight.copy_(rbm2.W.t())
            self.fc2.bias.copy_(rbm2.hb)

    def forward(self,x):
        x = x.view(-1,784)
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return self.fc3(x)

model = DBN().to(device)

# ---------------- FINETUNE ----------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr_ft)

train_accs, test_accs = [], []

for epoch in range(finetune_epochs):
    model.train()
    correct, total = 0,0

    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

        pred = out.argmax(1)
        correct += (pred==y).sum().item()
        total += y.size(0)

    train_acc = correct/total

    model.eval()
    correct,total = 0,0
    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)

    test_acc = correct/total

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1} | Train Acc={train_acc:.4f} | Test Acc={test_acc:.4f}")

# ---------------- PLOT ----------------
plt.plot(train_accs,label="Train")
plt.plot(test_accs,label="Test")
plt.legend()
plt.title("DBN Accuracy")
plt.show()